##### Setup
- Load the cleaned data (`df_clean_part*.parquet` from `03_clean.ipynb`) and split it into train/test sets for modeling

In [2]:
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split

CLEAN = "read_parquet('df_clean_part*.parquet')"

##### Why Split by Customer, Not by Row
- Already flagged in `01_eda.ipynb` (`customer-overlap-01`): the same `sha2_hash` appears in up to 11 monthly rows (11 months of data, Feb-Dec 2023). Most customers (1,950,872 of 2,175,327) appear in all 4 quarters
- A naive random **row-level** split would put different months of the *same customer* into both train and test — the model could learn customer-specific patterns from train and then get an inflated score by recognizing the same customer in test, rather than genuinely generalizing to unseen customers
- Fix: split at the **customer level** (`sha2_hash`) — every row belonging to a given customer goes entirely into either train or test, never both

In [ ]:
# One row per customer: did this customer ever show cancel_yn='\ud574\uc9c0' (cancelled) in any month?
# Used to stratify the split so both train/test preserve the same churn rate
customer_labels = duckdb.sql(f"""
    SELECT
        sha2_hash,
        max(CASE WHEN cancel_yn = '\ud574\uc9c0' THEN 1 ELSE 0 END) AS ever_cancelled
    FROM {CLEAN}
    GROUP BY sha2_hash
""").df()

print(f"n_customers: {len(customer_labels):,}")
customer_labels['ever_cancelled'].value_counts(normalize=True)

##### Stratified Customer-Level Split (80/20)
- `stratify=ever_cancelled` keeps the churn rate roughly equal between train and test
- `random_state=42` for reproducibility

In [ ]:
train_customers, test_customers = train_test_split(
    customer_labels,
    test_size=0.2,
    stratify=customer_labels['ever_cancelled'],
    random_state=42,
)

print(f"train customers: {len(train_customers):,}")
print(f"test customers:  {len(test_customers):,}")

##### Save the Split Assignment
- Only save `sha2_hash -> split` (small, ~2.2M rows) instead of duplicating the full ~1.3GB cleaned dataset into two copies
- Downstream modeling code reads `df_clean_part*.parquet` and joins against this split table to filter rows

In [ ]:
customer_split = pd.concat([
    train_customers.assign(split='train'),
    test_customers.assign(split='test'),
])[['sha2_hash', 'split']]

customer_split.to_parquet('customer_split.parquet', index=False)
print(f"saved customer_split.parquet ({len(customer_split):,} rows)")

##### Verify: No Customer Overlap, Similar Churn Rate
- Confirm no `sha2_hash` appears in both train and test
- Compare row-level `cancel_yn` rate between the two splits (should be close, thanks to stratification)

In [ ]:
verify_split = duckdb.sql(f"""
    SELECT
        cs.split,
        count(*) AS n_rows,
        count(DISTINCT c.sha2_hash) AS n_customers,
        round(count(*) FILTER (WHERE c.cancel_yn = '\ud574\uc9c0') * 100.0 / count(*), 2) AS pct_cancel_rows
    FROM {CLEAN} c
    JOIN customer_split cs USING (sha2_hash)
    GROUP BY cs.split
""").df()

verify_split

In [ ]:
n_overlap = duckdb.sql("""
    SELECT count(*) AS n_customers_in_both
    FROM (
        SELECT sha2_hash FROM customer_split WHERE split = 'train'
        INTERSECT
        SELECT sha2_hash FROM customer_split WHERE split = 'test'
    )
""").df()

n_overlap